# Book Recommender System

This notebook builds two recommendation models:
1. Popularity-based recommendations for books with strong ratings and enough readers.
2. Collaborative filtering recommendations based on similar user rating patterns.

The data comes from `books.csv`, `users.csv`, and `ratings.csv`.

In [1]:
# Main libraries used for data handling
import numpy as np
import pandas as pd

## Load Data

The project uses three CSV files:
- `books.csv`: book titles, authors, publishers, and image URLs.
- `users.csv`: user location and age details.
- `ratings.csv`: user ratings for books.

In [2]:
# Load all three datasets
books = pd.read_csv('books.csv')
users = pd.read_csv('users.csv')
ratings = pd.read_csv('ratings.csv')

C:\Users\Nitish\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


## Explore Data

Start with sample rows, dataset sizes, missing values, and duplicate rows.

In [120]:
books['Image-URL-M'][1]

'http://images.amazon.com/images/P/0002005018.01.MZZZZZZZ.jpg'

In [4]:
users.head()

In [5]:
ratings.head()

In [6]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(271360, 8)
(1149780, 3)
(278858, 3)


## Missing Values

Check which columns have missing data before building the models.

In [7]:
books.isnull().sum()

In [8]:
users.isnull().sum()

In [9]:
ratings.isnull().sum()

## Duplicate Rows

In [10]:
books.duplicated().sum()

In [15]:
ratings.duplicated().sum()

In [16]:
users.duplicated().sum()

## Popularity-Based Recommendations

This model recommends books that have both a high average rating and a strong number of ratings. A minimum rating count avoids promoting books that only have a few high scores.

The workflow is simple:
1. Merge ratings with book details using `ISBN`.
2. Count ratings for each book.
3. Calculate the average rating for each book.
4. Keep books with at least 250 ratings.
5. Sort by average rating and keep the top 50 books.

In [22]:
# Add book details to each rating using ISBN
ratings_with_name = ratings.merge(books, on='ISBN')

In [29]:
# Count how many ratings each book received
num_rating_df = ratings_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_df.rename(columns={'Book-Rating': 'num_ratings'}, inplace=True)
num_rating_df

In [30]:
# Calculate the average rating for each book
avg_rating_df = ratings_with_name.groupby('Book-Title').mean()['Book-Rating'].reset_index()
avg_rating_df.rename(columns={'Book-Rating': 'avg_rating'}, inplace=True)
avg_rating_df

In [31]:
popular_df = num_rating_df.merge(avg_rating_df, on='Book-Title')
popular_df

In [35]:
# Keep books with enough ratings, then take the top 50
popular_df = popular_df[popular_df['num_ratings'] >= 250].sort_values('avg_rating', ascending=False).head(50)

In [42]:
# Keep one row per book title with the fields needed by the app
popular_df = popular_df.merge(books, on='Book-Title').drop_duplicates('Book-Title')[['Book-Title', 'Book-Author', 'Image-URL-M', 'num_ratings', 'avg_rating']]

In [128]:
popular_df['Image-URL-M'][0]

'http://images.amazon.com/images/P/0439136350.01.MZZZZZZZ.jpg'

## Collaborative Filtering Recommendations

This model finds books with similar rating patterns. If two books are rated similarly by active readers, they are treated as related.

The raw data is sparse, so the model uses two filters:
- Users who rated more than 200 books.
- Books that received at least 50 ratings from those active users.

This keeps the similarity matrix smaller and more useful.

In [50]:
# Select users who have rated more than 200 books
x = ratings_with_name.groupby('User-ID').count()['Book-Rating'] > 200
padhe_likhe_users = x[x].index

In [54]:
# Keep ratings from those active users
filtered_rating = ratings_with_name[ratings_with_name['User-ID'].isin(padhe_likhe_users)]

In [59]:
# Keep books that have enough ratings from active users
y = filtered_rating.groupby('Book-Title').count()['Book-Rating'] >= 50
famous_books = y[y].index

In [63]:
final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]

In [69]:
# Build the book-user rating matrix
pt = final_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')

In [71]:
# Replace missing ratings with 0 for cosine similarity
pt.fillna(0, inplace=True)

In [72]:
pt

In [74]:
from sklearn.metrics.pairwise import cosine_similarity

In [77]:
# Compare each book with every other book
similarity_scores = cosine_similarity(pt)

In [80]:
similarity_scores.shape

(706, 706)

## Recommendation Function

The function returns four books that are closest to the selected title in the similarity matrix.

In [173]:
def recommend(book_name):
    index = np.where(pt.index == book_name)[0][0]

    # Skip the first result because it is the same book
    similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:5]

    data = []
    for i in similar_items:
        item = []
        temp_df = books[books['Book-Title'] == pt.index[i[0]]]

        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))

        data.append(item)

    return data

In [175]:
recommend('1984')

[['Animal Farm',
  'George Orwell',
  'http://images.amazon.com/images/P/0451526341.01.MZZZZZZZ.jpg'],
 ["The Handmaid's Tale",
  'Margaret Atwood',
  'http://images.amazon.com/images/P/0449212602.01.MZZZZZZZ.jpg'],
 ['Brave New World',
  'Aldous Huxley',
  'http://images.amazon.com/images/P/0060809833.01.MZZZZZZZ.jpg'],
 ['The Vampire Lestat (Vampire Chronicles, Book II)',
  'ANNE RICE',
  'http://images.amazon.com/images/P/0345313860.01.MZZZZZZZ.jpg']]

In [98]:
pt.index[545]

"The Handmaid's Tale"

## Save Model Files

Save the prepared data and similarity scores so the Flask app can load them without rebuilding the model each time.

In [127]:
import pickle

# Save popular books for the home page
pickle.dump(popular_df, open('popular.pkl', 'wb'))

In [146]:
books.drop_duplicates('Book-Title')

In [176]:
# Save files used by the Flask recommendation app
pickle.dump(pt, open('pt.pkl', 'wb'))
pickle.dump(books, open('books.pkl', 'wb'))
pickle.dump(similarity_scores, open('similarity_scores.pkl', 'wb'))